In [1]:
import os, json
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from langchain_core.tools import tool
from datetime import datetime, date
import uuid

load_dotenv()

True

In [2]:
with open("bajaj_db.json") as f:
    db = json.load(f)

@tool
def get_loan_status(loan_id):
    """Fetches current status of a Bajaj Finance loan from the database.
    Returns EMI amount, remaining months, outstanding balance, next due date.
    Use this when customer asks about their loan details, EMI, or balance.
    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    if loan_id not in db['loans']:
        return {"error": "Loan not found"}
    loan = db['loans'][loan_id]
    return {
        "customer_name": loan["customer_name"],
        "emi": loan["emi"],
        "remaining_months": loan["remaining_months"],
        "outstanding": loan["outstanding"],
        "next_due_date": loan["next_due_date"],
        "prepayment_charge_pct": loan["prepayment_charge_pct"]
    }

@tool
def get_emi_schedule(loan_id: str) -> dict:
    """Returns the upcoming EMI schedule (next 6 months) for a loan.

    Use this when the customer asks about their payment schedule,
    upcoming EMI dates, or future dues.

    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
    """
    loan_id = loan_id.strip().upper()

    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found."}

    loan = db["loans"][loan_id]
    emi = loan["emi_amount"]
    next_due = datetime.strptime(loan["next_due_date"], "%Y-%m-%d")

    schedule = []
    for i in range(min(6, loan["remaining_months"])):
        month = (next_due.month + i - 1) % 12 + 1
        year  = next_due.year + (next_due.month + i - 1) // 12
        due_date = date(year, month, next_due.day)
        schedule.append({
            "installment_no": loan["paid_months"] + i + 1,
            "due_date":       str(due_date),
            "amount":         emi,
        })

    return {
        "loan_id":          loan_id,
        "customer_name":    loan["customer_name"],
        "emi_amount":       emi,
        "remaining_months": loan["remaining_months"],
        "upcoming_schedule": schedule,
    }


@tool
def calculate_prepayment(loan_id: str, prepay_amount: float) -> dict:
    """Calculates prepayment penalty and net financial benefit for a loan.

    Use this when the customer asks about foreclosure, prepayment charges,
    or wants to know if making a prepayment is financially worth it.

    Args:
        loan_id:       The loan account number (e.g., 'BFL2024001')
        prepay_amount: Amount in ₹ the customer wants to prepay
    """
    loan_id = loan_id.strip().upper()

    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found."}

    loan        = db["loans"][loan_id]
    outstanding = loan["outstanding_balance"]
    rate        = loan["interest_rate"]
    remaining   = loan["remaining_months"]

    if prepay_amount > outstanding:
        return {
            "error":       f"Prepayment ₹{prepay_amount:,.0f} exceeds outstanding ₹{outstanding:,.0f}.",
            "max_prepay":  outstanding,
        }

    # 4% penalty if paid within first 3 years, 2% after
    charge_pct    = 4.0 if loan["paid_months"] < 36 else 2.0
    penalty       = prepay_amount * charge_pct / 100
    interest_saved = prepay_amount * (rate / 12 / 100) * remaining

    return {
        "loan_id":              loan_id,
        "prepay_amount":        prepay_amount,
        "penalty_percent":      charge_pct,
        "penalty_amount":       round(penalty, 2),
        "interest_saved":       round(interest_saved, 2),
        "net_benefit":          round(interest_saved - penalty, 2),
        "total_to_pay":         round(prepay_amount + penalty, 2),
        "is_full_foreclosure":  prepay_amount >= outstanding,
        "recommendation":       (
            "Prepayment is financially beneficial."
            if interest_saved > penalty else
            "Interest saved is less than penalty. Consider waiting."
        ),
    }


@tool
def process_refund_request(loan_id: str, reason: str, amount: float) -> dict:
    """Raises a refund request for a customer (excess EMI, double deduction, etc).

    Use this when the customer explicitly asks to raise a refund,
    file a dispute, or report a wrong deduction.

    Args:
        loan_id: The loan account number (e.g., 'BFL2024001')
        reason:  Reason for the refund request
        amount:  Amount in ₹ the customer is claiming as refund
    """
    loan_id = loan_id.strip().upper()

    if loan_id not in db["loans"]:
        return {"error": f"Loan {loan_id} not found."}

    ticket_id = "REF" + str(uuid.uuid4())[:8].upper()

    # you can store this ticket to json files --> send it to support team via email ??
    # SMTP --> you can easily 

    return {
        "success":             True,
        "ticket_id":           ticket_id,
        "loan_id":             loan_id,
        "customer_name":       db["loans"][loan_id]["customer_name"],
        "amount_claimed":      amount,
        "reason":              reason,
        "status":              "Raised",
        "expected_resolution": "5–7 working days",
        "message":             f"Your refund request has been logged. Ticket ID: {ticket_id}",
    }




In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm.invoke('what is the top 5 news today from Ind')

# how you can attach google or SerpAPI to llm
# then --> tool --> llm--> response

AIMessage(content="I'm unable to provide real-time news updates or current events as my knowledge was last updated in October 2021, and I don't have access to live data. However, you can check reliable news websites or apps for the latest news from India, such as The Times of India, The Hindu, or NDTV for up-to-date information.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 17, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_db286bd4bf', 'id': 'chatcmpl-E1jHgZ0s5lm1FARqy0Ko7Txuf5hpu', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f6385-4cb5-7922-b15b-706eba8b59b3-0', tool_calls=[], invalid_tool_calls=[], usage_

In [3]:
tools = [get_loan_status, get_emi_schedule, calculate_prepayment, process_refund_request]

In [4]:
# get_loan_status.invoke("BFL2024001")

In [5]:
# llm = ChatOpenAI(model="gpt-4o-mini")
# dummy= llm.invoke('what is my emi for loan BFL9988?')

In [6]:
# dummy.tool_calls

In [7]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(tools)



system = SystemMessage(content="You are a Bajaj Finance support agent. Use tools for real data.")
user_msg = HumanMessage(content="I got charged twice please intiate my refund for loan BFL9988")
message = [system, user_msg]
response = llm_with_tools.invoke(message)


In [8]:
response

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 409, 'total_tokens': 438, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_2247163164', 'id': 'chatcmpl-E1jEDOdxLbEqsacaCIvuN1wIc1BWq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f6382-065d-7080-9ab1-93f80950fdc5-0', tool_calls=[{'name': 'process_refund_request', 'args': {'loan_id': 'BFL9988', 'reason': 'Double deduction', 'amount': 0}, 'id': 'call_iIybvRVp4ic3zOlnpVrv7jTH', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 409, 'output_tokens': 29, 'total_tokens': 438, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output

In [9]:
response.tool_calls

[{'name': 'process_refund_request',
  'args': {'loan_id': 'BFL9988', 'reason': 'Double deduction', 'amount': 0},
  'id': 'call_iIybvRVp4ic3zOlnpVrv7jTH',
  'type': 'tool_call'}]

In [16]:
# response.tool_calls

In [11]:
tool_map = {
    "get_loan_status":get_loan_status,
    "get_emi_schedule":get_emi_schedule,
    "calculate_prepayment":calculate_prepayment,
    "process_refund_request":process_refund_request
}

tool_result = []

for tc in response.tool_calls:
    tool_name = tc["name"]
    tool_args = tc["args"]
    tool_id = tc["id"]
    result = tool_map[tool_name].invoke(tool_args)
    tool_result.append((tool_id,tool_name,result))
    

In [12]:
tool_result

[('call_elQUbpKLgnn6kCCjwWu4jmkb',
  'calculate_prepayment',
  {'loan_id': 'BFL9988',
   'prepay_amount': 5000.0,
   'penalty_percent': 2.0,
   'penalty_amount': 100.0,
   'interest_saved': 7437.5,
   'net_benefit': 7337.5,
   'total_to_pay': 5100.0,
   'is_full_foreclosure': False,
   'recommendation': 'Prepayment is financially beneficial.'})]

In [12]:
message = [system, user_msg,response]

for tool_id, tool_name, result in tool_result:
    tool_msg = ToolMessage(
        content=str(result),
        tool_call_id=tool_id
    )
    message.append(tool_msg)

final_response = llm_with_tools.invoke(message)
print("final answer:",final_response.content)

final answer: Your refund request for loan BFL9988 due to double deduction has been successfully logged. 

- **Ticket ID**: REFC1471A56
- **Status**: Raised
- **Expected Resolution**: 5–7 working days

If you have any further questions or need assistance, feel free to ask!


[SystemMessage(content='You are a Bajaj Finance support agent. Use tools for real data.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='what is my emi for loan BFL2024001?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 130, 'total_tokens': 151, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_b245e7bf62', 'id': 'chatcmpl-E1iiVN7CYFh6UeK3DGnZ7RkTSTi8I', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f6364-05f5-7350-8781-53d4a2057206-0', tool_calls=[{'name': 'get_loan_status', 'args': {'loan_id': 'BFL2024001'}, 'id': 'call_6dF8I254FzmenOqKuY

In [41]:
final_response.content

'Your EMI for loan BFL2024001 is ₹8,450. If you have any more questions or need further assistance, feel free to ask!'